# 🔀 Conditional Workflow — Challenge

## Scenario

AtlasTech Maroc's finance team receives dozens of employee expense claims every week.
Today they all land in one inbox and a human reads every single one.

Your task: build an automated **expense claim triage workflow** that:

1. **Checks** the claim against company policy (per-diem caps, hotel limits, submission deadlines)
2. **Routes** it to the right outcome — three possible paths:

| Policy decision | Path |
|-----------------|------|
| ✅ Fully compliant | → **Approval agent** drafts a confirmation email |
| ❓ Missing info | → Immediate output: clarification message to employee |
| ❌ Exceeds policy | → Immediate output: rejection notice with policy reference |

```
[Expense Claim]
    → PolicyCheckerAgent
    → to_policy_decision  (executor: parse JSON → ClaimDecision)
    → route_claim         (3-way routing function)
       ├─ to_approval_agent  → ApprovalDrafterAgent
       ├─ to_clarification   → (terminal output)
       └─ to_rejection       → (terminal output)
```

---

## ⏱ Time Budget — ~10 minutes

---

## YOUR TASK

The boilerplate (imports, agent creation, sample claim, terminal executors) is provided.  
Complete the **five steps** marked with `# TODO`:

| Step | What you need to do |
|------|---------------------|
| 1 | Write **agent instructions** for `PolicyCheckerAgent` and `ApprovalDrafterAgent` |
| 2 | Define the **`PolicyCheckerOutput`** Pydantic model and **`ClaimDecision`** dataclass |
| 3 | Implement the **`to_policy_decision`** executor (parse JSON → send `ClaimDecision`) |
| 4 | Implement the **`route_claim`** 3-way routing function |
| 5 | Build the **workflow** using `WorkflowBuilder` with correct edges |

> 💡 **Hints**:
> - `PolicyCheckerAgent` must return JSON with fields `status`, `reason`, `claim_summary`.
> - `status` must be one of: `"approved"`, `"clarification_needed"`, `"rejected"`.
> - The `to_approval_agent`, `to_clarification`, and `to_rejection` executors are already provided.
> - Use `.add_multi_selection_edge_group(source, [t1, t2, t3], selection_func=route_claim)`
>   for the 3-way branch, then `.add_edge(to_approval_agent, approval_agent)` for the one continuing path.

In [ ]:
####### Linux / MacOS Setup Instructions #######
# ! sudo apt update && sudo apt install graphviz -y   # Linux
# ! brew install graphviz                              # macOS

In [ ]:
# ! pip install -r ../../../Installation/requirements.txt -U

In [ ]:
import os
from typing import cast
from dataclasses import dataclass
from typing_extensions import Literal
from pydantic import BaseModel

In [ ]:
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider
from agent_framework import (
    AgentExecutor,
    AgentResponse,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    executor,
    WorkflowViz
)

In [ ]:
load_dotenv()

In [ ]:
# ============================================================
# YOUR TASK 1: Write Agent Instructions
# ============================================================
# PolicyCheckerInstructions:
#   - Checks expense claims against AtlasTech Maroc policy:
#       * Domestic per diem: max 600 MAD/day
#       * Hotel: max 800 MAD/night in Morocco
#       * Submission deadline: within 5 business days of trip end
#   - Must return JSON with exactly these fields:
#       { "status": "approved" | "clarification_needed" | "rejected",
#         "reason": "<specific explanation>",
#         "claim_summary": "<one-line summary of the claim>" }
#
# ApprovalDrafterInstructions:
#   - Drafts a short professional approval email to the employee
#   - Confirms the claim is approved, summarises it, and states
#     that payment will be processed within 5 working days
#
# TODO: Replace the placeholder strings below
# ============================================================

PolicyCheckerInstructions = """TODO: Write policy checker instructions here"""

ApprovalDrafterInstructions = """TODO: Write approval drafter instructions here"""

In [ ]:
# Sample expense claim submitted by an AtlasTech Maroc employee
EXPENSE_CLAIM = """
Employee: Karim Alaoui, Senior Engineer
Trip: Client on-site visit, Marrakech, 14–16 March 2026 (2 nights)
Submission date: 20 March 2026

Expenses:
  - Hotel: 2 nights × 1,050 MAD = 2,100 MAD  (Hotel La Mamounia)
  - Meals & incidentals: 2 days × 590 MAD   = 1,180 MAD
  - Taxi Casablanca → airport              =   150 MAD

Total claimed: 3,430 MAD
"""

In [ ]:
# ============================================================
# YOUR TASK 2: Define Output Models
# ============================================================
# PolicyCheckerOutput (Pydantic BaseModel):
#   - status: Literal["approved", "clarification_needed", "rejected"]
#   - reason: str
#   - claim_summary: str
#
# ClaimDecision (dataclass) — used to pass data between executors:
#   - status: str
#   - reason: str
#   - claim_summary: str
#
# TODO: Define both classes below
# ============================================================

# TODO: class PolicyCheckerOutput(BaseModel): ...

# TODO: @dataclass class ClaimDecision: ...

In [ ]:
# ============================================================
# YOUR TASK 3: Implement the Policy Decision Executor
# ============================================================
# This executor receives the PolicyCheckerAgent's raw JSON response,
# parses it with PolicyCheckerOutput, and sends a ClaimDecision
# to the next step via ctx.send_message().
#
# TODO: Complete the function body
# ============================================================

@executor(id="to_policy_decision")
async def to_policy_decision(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    response_text = response.agent_response.text.strip()
    print(f"Policy checker raw response: {response_text}")

    # TODO: parse response_text using PolicyCheckerOutput.model_validate_json()
    # TODO: build a ClaimDecision and call ctx.send_message() with it
    pass


# ============================================================
# YOUR TASK 4: Implement the 3-Way Routing Function
# ============================================================
# route_claim receives a ClaimDecision and a list of three target IDs
# in this order: [to_approval_agent_id, to_clarification_id, to_rejection_id]
#
# Return the list containing only the matching target ID:
#   status == "approved"             → [to_approval_agent_id]
#   status == "clarification_needed" → [to_clarification_id]
#   status == "rejected"             → [to_rejection_id]
#
# TODO: Complete the function body
# ============================================================

def route_claim(decision: "ClaimDecision", target_ids: list[str]) -> list[str]:
    approval_id, clarification_id, rejection_id = target_ids
    # TODO: implement 3-way routing
    pass


# --- Provided executors (do not modify) ---

@executor(id="to_approval_agent")
async def to_approval_agent(decision: "ClaimDecision", ctx: WorkflowContext) -> None:
    """Approved path: forward claim details to the ApprovalDrafterAgent."""
    await ctx.send_message(
        AgentExecutorRequest(
            messages=[Message("user", text=(
                f"The following expense claim has been approved:\n"
                f"Summary: {decision.claim_summary}\n"
                f"Notes: {decision.reason}\n\n"
                f"Please draft the approval notification email."
            ))],
            should_respond=True
        )
    )


@executor(id="to_clarification")
async def to_clarification(decision: "ClaimDecision", ctx: WorkflowContext) -> None:
    """Clarification path: output a message asking the employee for more information."""
    await ctx.yield_output(
        f"⚠️ Clarification needed for claim '{decision.claim_summary}':\n{decision.reason}"
    )


@executor(id="to_rejection")
async def to_rejection(decision: "ClaimDecision", ctx: WorkflowContext) -> None:
    """Rejection path: output a rejection notice with the policy reason."""
    await ctx.yield_output(
        f"❌ Expense claim rejected: '{decision.claim_summary}'\nReason: {decision.reason}"
    )

In [ ]:
from IPython.display import SVG, display, HTML

class ExpenseWorkflowEvent(WorkflowEvent): ...

In [ ]:
async with (
    AzureCliCredential() as credential,
    AzureAIAgentsProvider(credential=credential) as provider,
):
    policy_agent_obj = None
    approval_agent_obj = None
    try:
        policy_agent_obj = await provider.create_agent(
            name="policy-checker-agent",
            instructions=PolicyCheckerInstructions,
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        )
        policy_checker_agent = AgentExecutor(policy_agent_obj, id="policy_checker_agent")

        approval_agent_obj = await provider.create_agent(
            name="approval-drafter-agent",
            instructions=ApprovalDrafterInstructions,
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        )
        approval_agent = AgentExecutor(approval_agent_obj, id="approval_agent")

        # ============================================================
        # YOUR TASK 5: Build the Workflow
        # ============================================================
        # Chain the steps using WorkflowBuilder:
        #
        #   policy_checker_agent
        #       → to_policy_decision          (executor)
        #       → [to_approval_agent          (3-way branch via add_multi_selection_edge_group)
        #           | to_clarification
        #           | to_rejection]
        #       → approval_agent              (only from to_approval_agent path)
        #
        # Use:
        #   WorkflowBuilder(start_executor=policy_checker_agent)
        #   .add_edge(source, target)
        #   .add_multi_selection_edge_group(
        #       source=to_policy_decision,
        #       targets=[to_approval_agent, to_clarification, to_rejection],
        #       selection_func=route_claim
        #   )
        #   .add_edge(to_approval_agent, approval_agent)
        #   .build()
        #
        # TODO: Build and assign `workflow` below
        # ============================================================

        workflow = None  # TODO: replace with WorkflowBuilder(...).build()

        # Visualise (provided)
        print("Workflow visualization:")
        viz = WorkflowViz(workflow)
        print(viz.to_mermaid())
        svg_file = viz.export(format="svg")
        if svg_file and os.path.exists(svg_file):
            try:
                display(SVG(filename=svg_file))
            except Exception:
                with open(svg_file, "r", encoding="utf-8") as f:
                    display(HTML(f.read()))

        # Run the workflow
        task = (
            "Please review the following expense claim submitted by an AtlasTech Maroc employee "
            "and check whether it complies with company policy.\n\n" + EXPENSE_CLAIM
        )

        events = await workflow.run(task)

        # outputs can be plain strings (from yield_output) or AgentResponse objects
        for output in events.get_outputs():
            if isinstance(output, str):
                print(output)
            else:
                print(f"{output.messages[0].author_name}: {output.text}\n")

        print("Final state:", events.get_final_state())

    finally:
        for agent_obj in [policy_agent_obj, approval_agent_obj]:
            if agent_obj is not None:
                try:
                    await provider._agents_client.delete_agent(agent_obj.id)
                except Exception:
                    pass
        print("done")

---

## ⭐ OPTIONAL TASK: Add a 4th Routing Path — Manager Escalation

Extend the workflow with a **Manager Escalation** path for claims above 5,000 MAD total.
High-value claims bypass normal approval and go straight to the manager.

**Steps:**
1. Add `"escalated"` to the `status` Literal in `PolicyCheckerOutput` and update the policy checker instructions to set `status = "escalated"` when the total claim exceeds 5,000 MAD.
2. Add an `@executor(id="to_escalation")` that calls `ctx.yield_output()` with an escalation notice.
3. Update `route_claim` to unpack 4 target IDs and handle `"escalated"` → `[escalation_id]`.
4. Add `to_escalation` as the 4th target in `add_multi_selection_edge_group`.

Test with a claim total above 5,000 MAD and verify the escalation path fires.